In [ ]:
from google.colab import files
uploaded = files.upload()

import os
os.rename("gentle-complex-473322-d9-b842c9617a14.json", "service-account.json")
# =====================================
# 🔧 CONFIGURACIÓN INICIAL
# =====================================
!pip install google-cloud-pubsub psycopg2-binary --quiet

import time
import random
import json
from datetime import datetime
from google.cloud import pubsub_v1

# -------------------------------------
# Autenticación con cuenta de servicio
# -------------------------------------
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/service-account.json"  # ⚠️ Ajusta la ruta

# -------------------------------------
# Configuración de Pub/Sub
# -------------------------------------
project_id = "gentle-complex-473322-d9"
topic_id = "datos_dispositivos"
publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(project_id, topic_id)

# =====================================
# 🌡️ GENERADOR DE DATOS SIMULADOS
# =====================================

last_values = {
    "tem": None, "hum": None, "pres": None,
    "mp1_0_stp": None, "mp2_5_stp": None, "mp10_stp": None,
    "mp1_0_ate": None, "mp2_5_ate": None, "mp10_ate": None,
    "co2": None, "dir_viento": None, "rap_viento": None,
    "consumo_1": None, "consumo_2": None, "consumo_3": None,
    "agua_caida": None
}

def suavizar(nombre, esperado, variacion=1.0):
    if last_values[nombre] is None:
        val = esperado + random.uniform(-variacion, variacion)
    else:
        val = last_values[nombre] + random.uniform(-variacion, variacion)
        val = (val + esperado) / 2
    last_values[nombre] = val
    return val

def generar_datos():
    ahora = datetime.now()
    hora = ahora.hour
    mes = ahora.month
    dia_semana = ahora.weekday()

    # --- Temperatura y Humedad ---
    if mes in [12,1,2]: t_min, t_max = 14, 32
    elif mes in [6,7,8]: t_min, t_max = 5, 18
    else: t_min, t_max = 10, 25

    if hora < 15:
        frac = hora/15
        temp_esperada = t_min + (t_max - t_min) * (frac**1.5)
    else:
        frac = (hora-15)/9
        temp_esperada = t_max - (t_max - t_min) * (frac**0.7)

    tem_bme280 = round(suavizar("tem", temp_esperada, 0.3), 1)
    hum_esperada = 90 - (tem_bme280 - t_min)*2
    hum_bme280 = max(5, round(suavizar("hum", hum_esperada, 1), 1))
    pres_bme280 = round(suavizar("pres", 1010, 0.1) + random.uniform(-1,1), 1)
    alt_bme280 = 570.0

    # --- Material particulado ---
    factor_trafico = 1 if (7 <= hora <= 9 or 18 <= hora <= 21) and dia_semana < 5 else 0
    factor_invierno = 1.5 if mes in [5,6,7,8] else 1.0
    mp25_esperado = 15 + 20*factor_trafico*factor_invierno
    mp2_5_stp = round(suavizar("mp2_5_stp", mp25_esperado, 1.5), 1)
    mp10_stp = round(suavizar("mp10_stp", mp2_5_stp*1.2, 2), 1)
    mp1_0_stp = round(suavizar("mp1_0_stp", mp2_5_stp*0.7, 1), 1)
    mp1_0_ate = round(suavizar("mp1_0_ate", mp1_0_stp, 2), 1)
    mp2_5_ate = round(suavizar("mp2_5_ate", mp2_5_stp, 3), 1)
    mp10_ate = round(suavizar("mp10_ate", mp10_stp, 3), 1)
    mp_gt_03 = mp2_5_stp*random.uniform(50,200)
    mp_gt_05 = mp2_5_stp*random.uniform(20,100)
    mp_gt_1 = mp2_5_stp*random.uniform(2,20)
    mp_gt_25 = mp2_5_stp*random.uniform(0.5,5)
    mp_gt_5 = mp2_5_stp*random.uniform(0.2,2)
    mp_gt_10 = mp2_5_stp*random.uniform(0.1,1)

    # --- CO2 ---
    co2_esperado = 420 + mp2_5_stp*2
    if mes in [6,7,8]: co2_esperado += 200
    co2_mhz19 = round(suavizar("co2", co2_esperado, 5), 1)

    # --- Viento ---
    dir_viento = round(suavizar("dir_viento", random.uniform(0,360), 5), 1)
    base_viento = 2 if hora < 12 else 5
    rap_viento = round(suavizar("rap_viento", base_viento, 0.5), 1)

    # --- Agua caída y consumo ---
    agua_exp = random.uniform(0.2, 0.8) if 0 <= hora <= 6 else random.uniform(0, 0.4)
    agua_caida = round(suavizar("agua_caida", agua_exp, 0.05), 2)
    consumo_1 = round(suavizar("consumo_1", 50, 5), 1)
    consumo_2 = round(suavizar("consumo_2", 20, 3), 1)
    consumo_3 = round(suavizar("consumo_3", 10, 2), 1)

    return {
        "tem_bme280": tem_bme280, "hum_bme280": hum_bme280, "pres_bme280": pres_bme280, "alt_bme280": alt_bme280,
        "mp1.0_stp": mp1_0_stp, "mp2.5_stp": mp2_5_stp, "mp10_stp": mp10_stp,
        "mp1.0_ate": mp1_0_ate, "mp2.5_ate": mp2_5_ate, "mp10_ate": mp10_ate,
        "mp_gt_0.3um": mp_gt_03, "mp_gt_0.5um": mp_gt_05, "mp_gt_1.0um": mp_gt_1, "mp_gt_2.5um": mp_gt_25,
        "mp_gt_5.0um": mp_gt_5, "mp_gt_10um": mp_gt_10, "co2_mhz19": co2_mhz19,
        "dir_viento": dir_viento, "rap_viento": rap_viento, "agua_caida": agua_caida,
        "consumo_1": consumo_1, "consumo_2": consumo_2, "consumo_3": consumo_3, "id_dispositivo": 1
    }

# =====================================
# 🚀 PUBLICADOR EN PUB/SUB
# =====================================
try:
    while True:
        data_dict = generar_datos()
        data_json = json.dumps(data_dict)
        publisher.publish(topic_path, data_json.encode("utf-8"))
        print(f"{datetime.now()} -> 📤 Enviado a Pub/Sub: {data_json}")
        time.sleep(60)  # Enviar cada 60 segundos
except KeyboardInterrupt:
    print("Simulación detenida manualmente.")